In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import json

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

pd.set_option("display.max_columns", 100)

In [2]:
PROJECT_ROOT = Path("..")

FEATURE_DATA_PATH = PROJECT_ROOT / "01_data" / "features" / "sales_features.csv"

MODEL_DIR = PROJECT_ROOT / "04_models"
MODEL_RESULTS_DIR = PROJECT_ROOT / "05_outputs" / "model_results"
FORECAST_DIR = PROJECT_ROOT / "05_outputs" / "forecasts"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FORECAST_DIR.mkdir(parents=True, exist_ok=True)

print(FEATURE_DATA_PATH)

..\01_data\features\sales_features.csv


In [3]:
df = pd.read_csv(FEATURE_DATA_PATH)

df["shipped_date"] = pd.to_datetime(df["shipped_date"])

print("Shape:", df.shape)
print("Date range:", df["shipped_date"].min(), "to", df["shipped_date"].max())
print("Unique SKUs:", df["sku"].nunique())

df.head()

Shape: (26484, 36)
Date range: 2021-01-29 00:00:00 to 2022-01-02 00:00:00
Unique SKUs: 180


,shipped_date,sku,sku_encoded,qty,qty_log,year,month,day,day_of_week,week_of_year,quarter,is_weekend,revenue,cost_of_good_sold,moq_order,channel_count,qty_lag_1,qty_lag_2,qty_lag_3,qty_lag_7,qty_lag_14,qty_rolling_mean_3,qty_rolling_std_3,qty_rolling_min_3,qty_rolling_max_3,qty_rolling_mean_7,qty_rolling_std_7,qty_rolling_min_7,qty_rolling_max_7,qty_rolling_mean_14,qty_rolling_std_14,qty_rolling_min_14,qty_rolling_max_14,sku_expanding_mean_qty,sku_expanding_median_qty,sku_expanding_std_qty
0,2021-02-04,089A0E,0,20,3.044522,2021,2,4,3,5,1,0,529.2,428.65,56460,1,300.0,110.0,90.0,60.0,220.0,166.666667,115.902258,90.0,300.0,121.428571,92.992575,40.0,300.0,128.571429,76.445772,40.0,300.0,128.571429,125.0,76.445772
1,2021-02-06,089A0E,0,60,4.110874,2021,2,6,5,5,1,1,1587.6,1285.96,56460,1,20.0,300.0,110.0,190.0,140.0,143.333333,142.945211,20.0,300.0,115.714286,98.464400,20.0,300.0,114.285714,76.732732,20.0,300.0,121.333333,110.0,78.818659
2,2021-02-10,089A0E,0,40,3.713572,2021,2,10,2,6,1,0,1058.4,857.30,56460,1,60.0,20.0,300.0,60.0,150.0,126.666667,151.437556,20.0,300.0,97.142857,94.289322,20.0,300.0,108.571429,77.643876,20.0,300.0,117.500000,100.0,77.674535
3,2021-02-12,089A0E,0,100,4.615121,2021,2,12,4,6,1,0,2646.0,2143.26,56460,1,40.0,60.0,20.0,40.0,190.0,40.000000,20.000000,20.0,60.0,94.285714,95.891804,20.0,300.0,100.714286,78.687726,20.0,300.0,112.941176,90.0,77.521344
4,2021-02-14,089A0E,0,50,3.931826,2021,2,14,6,6,1,1,1323.0,1071.63,56460,1,100.0,40.0,60.0,90.0,60.0,66.666667,30.550505,40.0,100.0,102.857143,92.864469,20.0,300.0,94.285714,74.391303,20.0,300.0,112.222222,95.0,75.268582


In [4]:
df.columns.tolist()

['shipped_date',
 'sku',
 'sku_encoded',
 'qty',
 'qty_log',
 'year',
 'month',
 'day',
 'day_of_week',
 'week_of_year',
 'quarter',
 'is_weekend',
 'revenue',
 'cost_of_good_sold',
 'moq_order',
 'channel_count',
 'qty_lag_1',
 'qty_lag_2',
 'qty_lag_3',
 'qty_lag_7',
 'qty_lag_14',
 'qty_rolling_mean_3',
 'qty_rolling_std_3',
 'qty_rolling_min_3',
 'qty_rolling_max_3',
 'qty_rolling_mean_7',
 'qty_rolling_std_7',
 'qty_rolling_min_7',
 'qty_rolling_max_7',
 'qty_rolling_mean_14',
 'qty_rolling_std_14',
 'qty_rolling_min_14',
 'qty_rolling_max_14',
 'sku_expanding_mean_qty',
 'sku_expanding_median_qty',
 'sku_expanding_std_qty']

In [5]:
target_col = "qty_log"

exclude_cols = [
    "shipped_date",
    "sku",
    "qty",
    "qty_log",
    "revenue",
    "cost_of_good_sold",
    "channel_count"
]

feature_cols = [col for col in df.columns if col not in exclude_cols]

print("Number of features:", len(feature_cols))
print(feature_cols)

Number of features: 29
['sku_encoded', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend', 'moq_order', 'qty_lag_1', 'qty_lag_2', 'qty_lag_3', 'qty_lag_7', 'qty_lag_14', 'qty_rolling_mean_3', 'qty_rolling_std_3', 'qty_rolling_min_3', 'qty_rolling_max_3', 'qty_rolling_mean_7', 'qty_rolling_std_7', 'qty_rolling_min_7', 'qty_rolling_max_7', 'qty_rolling_mean_14', 'qty_rolling_std_14', 'qty_rolling_min_14', 'qty_rolling_max_14', 'sku_expanding_mean_qty', 'sku_expanding_median_qty', 'sku_expanding_std_qty']


In [6]:
unique_dates = np.sort(df["shipped_date"].unique())

split_idx = int(len(unique_dates) * 0.8)
split_date = unique_dates[split_idx]

train_df = df[df["shipped_date"] < split_date].copy()
test_df = df[df["shipped_date"] >= split_date].copy()

print("Split date:", split_date)

print("\nTrain shape:", train_df.shape)
print("Train date range:", train_df["shipped_date"].min(), "to", train_df["shipped_date"].max())

print("\nTest shape:", test_df.shape)
print("Test date range:", test_df["shipped_date"].min(), "to", test_df["shipped_date"].max())

Split date: 2021-10-26T00:00:00.000000

Train shape: (21102, 36)
Train date range: 2021-01-29 00:00:00 to 2021-10-24 00:00:00

Test shape: (5382, 36)
Test date range: 2021-10-26 00:00:00 to 2022-01-02 00:00:00


In [7]:
X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (21102, 29)
X_test: (5382, 29)


In [9]:
def calculate_wape(y_true, y_pred):
    """
    WAPE = sum(abs(actual - predicted)) / sum(actual)
    Đây là metric rất dễ hiểu cho business.
    """
    denominator = np.sum(np.abs(y_true))
    
    if denominator == 0:
        return np.nan
    
    return np.sum(np.abs(y_true - y_pred)) / denominator


def evaluate_predictions(y_true_log, y_pred_log, model_name):
    """
    Convert log prediction back to original qty scale,
    then calculate MAE, RMSE, WAPE, R2.
    """
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    
    # Forecast quantity cannot be negative
    y_pred = np.clip(y_pred, 0, None)
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    wape = calculate_wape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return {
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "WAPE": wape,
        "WAPE_percent": wape * 100,
        "R2": r2
    }

In [10]:
baseline_pred_qty = test_df["qty_lag_1"].values
baseline_pred_qty = np.clip(baseline_pred_qty, 0, None)

baseline_pred_log = np.log1p(baseline_pred_qty)

baseline_result = evaluate_predictions(
    y_true_log=y_test,
    y_pred_log=baseline_pred_log,
    model_name="Baseline_Lag_1"
)

baseline_result

{'model': 'Baseline_Lag_1',
 'MAE': 250.893348197696,
 'RMSE': np.float64(821.5377615004456),
 'WAPE': np.float64(0.5020609548880287),
 'WAPE_percent': np.float64(50.20609548880287),
 'R2': 0.3037502987130265}

In [11]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred_log = rf_model.predict(X_test)

rf_result = evaluate_predictions(
    y_true_log=y_test,
    y_pred_log=rf_pred_log,
    model_name="Random_Forest"
)

rf_result

{'model': 'Random_Forest',
 'MAE': 228.09284769027911,
 'RMSE': np.float64(694.9119411092416),
 'WAPE': np.float64(0.45643503001233754),
 'WAPE_percent': np.float64(45.64350300123375),
 'R2': 0.5018392458109235}

In [12]:
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

xgb_pred_log = xgb_model.predict(X_test)

xgb_result = evaluate_predictions(
    y_true_log=y_test,
    y_pred_log=xgb_pred_log,
    model_name="XGBoost"
)

xgb_result

{'model': 'XGBoost',
 'MAE': 230.88311081471295,
 'RMSE': np.float64(704.8495166212339),
 'WAPE': np.float64(0.4620186063753835),
 'WAPE_percent': np.float64(46.20186063753835),
 'R2': 0.4874894925354508}

In [13]:
lgbm_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgbm_model.fit(X_train, y_train)

lgbm_pred_log = lgbm_model.predict(X_test)

lgbm_result = evaluate_predictions(
    y_true_log=y_test,
    y_pred_log=lgbm_pred_log,
    model_name="LightGBM"
)

lgbm_result

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002289 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5511
[LightGBM] [Info] Number of data points in the train set: 21102, number of used features: 28
[LightGBM] [Info] Start training from score 5.045070


{'model': 'LightGBM',
 'MAE': 227.56352305361443,
 'RMSE': np.float64(688.5776672607416),
 'WAPE': np.float64(0.455375802119535),
 'WAPE_percent': np.float64(45.5375802119535),
 'R2': 0.5108795427966486}

In [14]:
model_results = pd.DataFrame([
    baseline_result,
    rf_result,
    xgb_result,
    lgbm_result
])

model_results = model_results.sort_values("WAPE").reset_index(drop=True)

model_results

,model,MAE,RMSE,WAPE,WAPE_percent,R2
0,LightGBM,227.563523,688.577667,0.455376,45.537580,0.510880
1,Random_Forest,228.092848,694.911941,0.456435,45.643503,0.501839
2,XGBoost,230.883111,704.849517,0.462019,46.201861,0.487489
3,Baseline_Lag_1,250.893348,821.537762,0.502061,50.206095,0.303750


In [15]:
MODEL_COMPARISON_PATH = MODEL_RESULTS_DIR / "model_comparison.csv"

model_results.to_csv(MODEL_COMPARISON_PATH, index=False)

print(f"Saved model comparison to: {MODEL_COMPARISON_PATH}")

Saved model comparison to: ..\05_outputs\model_results\model_comparison.csv


In [16]:
best_model_name = model_results.loc[0, "model"]

print("Best model:", best_model_name)

model_dict = {
    "Random_Forest": rf_model,
    "XGBoost": xgb_model,
    "LightGBM": lgbm_model
}

prediction_dict = {
    "Baseline_Lag_1": baseline_pred_log,
    "Random_Forest": rf_pred_log,
    "XGBoost": xgb_pred_log,
    "LightGBM": lgbm_pred_log
}

# Nếu baseline thắng, mình vẫn lưu model ML tốt nhất tiếp theo,
# vì baseline không phải object model để deploy / explain bằng SHAP.
if best_model_name == "Baseline_Lag_1":
    best_ml_model_name = model_results[model_results["model"] != "Baseline_Lag_1"].iloc[0]["model"]
else:
    best_ml_model_name = best_model_name

best_model = model_dict[best_ml_model_name]
best_pred_log = prediction_dict[best_ml_model_name]

print("Best ML model saved:", best_ml_model_name)

Best model: LightGBM
Best ML model saved: LightGBM


In [17]:
BEST_MODEL_PATH = MODEL_DIR / "best_sales_forecasting_model.pkl"
FEATURE_LIST_PATH = MODEL_DIR / "feature_columns.json"
BEST_MODEL_INFO_PATH = MODEL_DIR / "best_model_info.json"

joblib.dump(best_model, BEST_MODEL_PATH)

with open(FEATURE_LIST_PATH, "w") as f:
    json.dump(feature_cols, f, indent=4)

best_model_info = {
    "best_model_name": best_ml_model_name,
    "target": target_col,
    "trained_on_log_target": True,
    "split_date": str(split_date),
    "number_of_features": len(feature_cols)
}

with open(BEST_MODEL_INFO_PATH, "w") as f:
    json.dump(best_model_info, f, indent=4)

print(f"Saved model to: {BEST_MODEL_PATH}")
print(f"Saved feature list to: {FEATURE_LIST_PATH}")
print(f"Saved model info to: {BEST_MODEL_INFO_PATH}")

Saved model to: ..\04_models\best_sales_forecasting_model.pkl
Saved feature list to: ..\04_models\feature_columns.json
Saved model info to: ..\04_models\best_model_info.json


In [18]:
test_predictions = test_df[["shipped_date", "sku", "qty"]].copy()

test_predictions["actual_qty"] = test_predictions["qty"]

test_predictions["baseline_pred_qty"] = np.clip(np.expm1(baseline_pred_log), 0, None)
test_predictions["rf_pred_qty"] = np.clip(np.expm1(rf_pred_log), 0, None)
test_predictions["xgb_pred_qty"] = np.clip(np.expm1(xgb_pred_log), 0, None)
test_predictions["lgbm_pred_qty"] = np.clip(np.expm1(lgbm_pred_log), 0, None)
test_predictions["best_model_pred_qty"] = np.clip(np.expm1(best_pred_log), 0, None)

test_predictions["absolute_error"] = abs(
    test_predictions["actual_qty"] - test_predictions["best_model_pred_qty"]
)

test_predictions = test_predictions.drop(columns=["qty"])

test_predictions.head()

,shipped_date,sku,actual_qty,baseline_pred_qty,rf_pred_qty,xgb_pred_qty,lgbm_pred_qty,best_model_pred_qty,absolute_error
92,2021-10-26,089A0E,340,150.0,78.166620,71.288078,89.391441,89.391441,250.608559
93,2021-10-30,089A0E,690,340.0,96.802173,55.510487,71.103282,71.103282,618.896718
94,2021-11-01,089A0E,430,690.0,160.831643,116.924973,167.716792,167.716792,262.283208
95,2021-11-05,089A0E,1030,430.0,222.317310,156.735306,302.494836,302.494836,727.505164
96,2021-11-07,089A0E,1180,1030.0,196.341160,111.166679,209.411043,209.411043,970.588957


In [19]:
FORECAST_OUTPUT_PATH = FORECAST_DIR / "test_forecasts.csv"

test_predictions.to_csv(FORECAST_OUTPUT_PATH, index=False)

print(f"Saved test forecasts to: {FORECAST_OUTPUT_PATH}")
print("Shape:", test_predictions.shape)

Saved test forecasts to: ..\05_outputs\forecasts\test_forecasts.csv
Shape: (5382, 9)


In [20]:
sku_error = (
    test_predictions
    .groupby("sku")
    .agg(
        actual_qty=("actual_qty", "sum"),
        predicted_qty=("best_model_pred_qty", "sum"),
        absolute_error=("absolute_error", "sum")
    )
    .reset_index()
)

sku_error["wape"] = sku_error["absolute_error"] / sku_error["actual_qty"]
sku_error = sku_error.sort_values("absolute_error", ascending=False)

sku_error.head(20)

,sku,actual_qty,predicted_qty,absolute_error,wape
73,FJDJ6B,232470,128282.329997,152701.192079,0.656864
48,8LCPHF,77077,37930.189625,47720.489109,0.619127
165,XDYFFX,66582,25104.841409,47107.429116,0.707510
107,LHR5LZ,57876,21933.588437,42684.589114,0.737518
35,67LPLP,112056,95866.567241,36536.236173,0.326053
5,1JKADT,111230,95173.028009,34202.019215,0.307489
26,4OCITK,89094,81234.045940,26957.308452,0.302572
173,Y6HWKQ,59614,39849.802020,25419.344857,0.426399
89,HVARMV,55704,34522.376239,25056.798271,0.449820
122,NLDP86,49322,35282.620912,22936.094181,0.465028


In [21]:
SKU_ERROR_PATH = MODEL_RESULTS_DIR / "sku_error_analysis.csv"

sku_error.to_csv(SKU_ERROR_PATH, index=False)

print(f"Saved SKU error analysis to: {SKU_ERROR_PATH}")

Saved SKU error analysis to: ..\05_outputs\model_results\sku_error_analysis.csv


In [22]:
if hasattr(best_model, "feature_importances_"):
    feature_importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": best_model.feature_importances_
    }).sort_values("importance", ascending=False)
    
    display(feature_importance.head(20))
    
    FEATURE_IMPORTANCE_PATH = MODEL_RESULTS_DIR / "feature_importance.csv"
    feature_importance.to_csv(FEATURE_IMPORTANCE_PATH, index=False)
    
    print(f"Saved feature importance to: {FEATURE_IMPORTANCE_PATH}")
else:
    print("Best model does not have feature_importances_")

,feature,importance
4,day_of_week,881
12,qty_lag_7,812
9,qty_lag_1,785
13,qty_lag_14,749
15,qty_rolling_std_3,678
18,qty_rolling_mean_7,674
19,qty_rolling_std_7,638
3,day,631
8,moq_order,631
28,sku_expanding_std_qty,626


Saved feature importance to: ..\05_outputs\model_results\feature_importance.csv
